# Prompt 2 — LLM Generator (G)

This notebook isolates the **Generator prompt** used in cycles greater than 0.

The Generator receives:

1. **PARENT CMA-ES ALGORITHM** — the exact parent selected by the deterministic framework.
2. **EXPLAINABILITY REPORT** — the actionable report produced by the Explainer.
3. **CURRENT CYCLE** — the current optimization cycle.

Its task is to return a complete executable Python class named `LLM_CMAES` implementing only the single conceptual modification specified in `TRY`.


## 1. Generator prompt


In [ ]:
# ============================================================
# GENERATOR PROMPT - CYCLES > 0
# ============================================================

def cmaes_generation_prompt_cycle_n(
        parent_algorithm: str,
        explainability_report: str,
        cycle_num: int):

    return f"""
You are an expert in CMA-ES and evolutionary optimization.

GOAL

Generate the next complete CMA-ES variant as a Python class named LLM_CMAES.

INPUTS

(A) PARENT CMA-ES ALGORITHM:
{parent_algorithm}

(B) EXPLAINABILITY REPORT:
{explainability_report}

(C) CURRENT CYCLE:
{cycle_num}


STRICT EXECUTABLE OUTPUT CONTRACT

- The generated Python code MUST define a class named exactly LLM_CMAES.
- LLM_CMAES MUST implement both __init__ and __call__.
- Do not rename, alias, or conditionally define LLM_CMAES.
- The final optimizer used by the framework must be exposed through LLM_CMAES.
- Helper functions or helper classes are allowed, but every referenced helper
  must be defined inside the same generated Python code.
- Do not depend on scipy, pandas, sklearn, cma, modea, or external files.
- Use only Python standard functionality, numpy, and ioh.
- Return a self-contained executable implementation.
- Return Python code only.
- Do not return markdown fences.
- Do not include explanations outside the code.


GUIDANCE

The EXPLAINABILITY_REPORT is the primary source of guidance.

Implement only the modification specified in TRY.

Preserve everything listed in KEEP.

Remove everything listed in REMOVE.

Do not reintroduce mechanisms that the report has explicitly abandoned.

The PARENT CMA-ES ALGORITHM is the exact algorithm from which the new
variant must be derived. It may be the immediately previous variant,
the best stable historical variant, or the fixed baseline.


ONE-MODIFICATION RULE

Implement only ONE conceptual algorithmic modification.

Do not introduce additional mechanisms that are not explicitly requested
by the EXPLAINABILITY_REPORT.

A conceptual modification may require several code-line changes, but all
changes must implement the same single mechanism.

Do not simultaneously introduce independent changes to:

- sampling,
- step-size adaptation,
- covariance adaptation,
- mean recombination,
- population size,
- restart logic.


ALLOWED MODIFICATION AREAS

The recommended modification must be implemented primarily in one of:

1. LLM SECTION 1: Sampling
2. LLM SECTION 2: Mean update
3. LLM SECTION 3: Adaptation


FRAMEWORK REQUIREMENTS

- Return one complete Python class named LLM_CMAES.
- Use only numpy and ioh.
- Keep the constructor and __call__ interface compatible with the parent algorithm.
- Keep the main population evaluation step unchanged.
- Do not introduce additional objective-function evaluations.
- The number of calls to problem(...) per generation must remain identical
  to the parent/baseline implementation.
- If exploratory candidates or probes are used, they must REPLACE existing
  offspring inside the original lambda-sized population before evaluation.
- Never evaluate a probe, local-search point, or auxiliary candidate separately.
- Keep the total evaluation budget identical to the baseline.
- Keep candidate solutions inside the problem bounds.
- Preserve all numerical safeguards present in the PARENT CMA-ES ALGORITHM.
- Keep sigma finite and positive.
- Keep C finite, symmetric, and numerically stable.
- Keep the main optimization loop bounded.
- Do not create an additional LLM SECTION outside Sections 1, 2, and 3.


INTERFACE REQUIREMENTS

The generated class must keep this structure:

class LLM_CMAES:

    def __init__(
            self,
            pop_size=20,
            num_iterations=100,
            sigma=0.5
        ):
        ...

    def __call__(self, problem):
        ...

The __call__ method must return:

(best_solution, best_fitness)


VARIANT DESCRIPTION

The class must contain:

self.variant_description = {{
    "sampling": "...",
    "mean_update": "...",
    "adaptation": "..."
}}

Only the component modified in the current cycle should describe a new
mechanism. The other two entries should explicitly state that they remain
unchanged from the parent algorithm.


OUTPUT FORMAT

Return only the complete Python code.

Do not use markdown code fences.

Do not write explanations before or after the code.
"""


## 2. Prompt-selection helper


In [ ]:

def select_generation_prompt(
        parent_algorithm,
        explainability_report,
        cycle_num):

    if cycle_num <= 0:
        raise ValueError(
            "The Generator LLM is only used for cycles greater than 0."
        )

    print(f"Usando GENERATOR PROMPT para CYCLE {cycle_num}")

    return cmaes_generation_prompt_cycle_n(
        parent_algorithm=parent_algorithm,
        explainability_report=explainability_report,
        cycle_num=cycle_num
    )


## 3. Example input mapping

The following variables represent the three inputs supplied to the Generator.


In [ ]:
parent_algorithm = """<SELECTED PARENT CMA-ES CODE>"""

explainability_report = """
<EXPLAINABILITY REPORT FROM THE PREVIOUS EXPLAINER CALL>
"""

cycle_num = 1

generator_prompt_text = select_generation_prompt(
    parent_algorithm=parent_algorithm,
    explainability_report=explainability_report,
    cycle_num=cycle_num
)

# print(generator_prompt_text)
